## Max Rectangle - Matrix

Matrix - list of rows in python
    

min_h = float('inf') = start with "infinity" so any number is smaller

for k... = check each column in the range

if heights[k] < min_h = keep the smallest

width = ... = count columns

area = ... = multiply

if area > best = keep the best

┌─────────────────────────────────────┐ │ ROW LOOP (i) │ │ "Slide the floor down" │ │ │ │ ┌─────────────────────────────┐ │ │ │ COLUMN LOOP (j) │ │ │ │ "Update the bar" │ │ │ │ heights[j] grows or breaks │ │ │ └─────────────────────────────┘ │ │ │ │ [Heights now looks like: 2, 1, 3] │ │ │ │ ┌─────────────────────────────┐ │ │ │ RANGE LOOP (left, right) │ │ │ │ "Check this rectangle" │ │ │ │ - Find shortest bar │ │ │ │ - Count bars (width) │ │ │ │ - Area = short × count │ │ │ └─────────────────────────────┘ │ └─────────────────────────────────────┘

[Row Loop i] 
    ↓ 
updates [Heights Array] → 
                        [1, 2, 0, 2] 
    ↓ 
check ranges inside [Left=0, Right=3] → 
                                        shortest bar = ? → area = ?


```python
min_h = float('inf')
for k in range(left, right + 1):
    if heights[k] < min_h:
        min_h = heights[k]
width = right - left + 1
area = min_h * width
if area > best:
    best = area
```    
---

Updated todo list

You already have the histogram idea in place in max_rectangle.ipynb: the `heights` array is the histogram.

The rule is simple:

- Start with `heights = [0] * cols`
- For each row:
  - If the cell is `"1"`, increase that column’s height by 1
  - If the cell is `"0"`, reset that column’s height to 0

That means each position in `heights` becomes the number of consecutive 1s stacked upward ending at the current row.

Example:

```text
matrix rows:
1 0 1 0 0
1 0 1 1 1
1 1 1 1 1
1 0 0 1 0
```

After each row, the histogram heights become:

- After row 1: `[1, 0, 1, 0, 0]`
- After row 2: `[2, 0, 2, 1, 1]`
- After row 3: `[3, 1, 3, 2, 2]`
- After row 4: `[4, 0, 0, 3, 0]`

So the matrix problem turns into a repeated histogram problem: for each row, treat `heights` like bar heights and find the biggest rectangle you can form there.

The key mental model is this: each row is the “floor,” and `heights` tells you how tall each column is if you stack 1s upward from that floor.

If you want, I can next explain how to get the max rectangle from one histogram step by step using your current example.

*   The column number (start_col)
*   The shortest bar value (min_h)


In [7]:
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        # So the update rule is: if current cell is 1, height = previous row's height + 1. If it's 0, height = 0.
        if not matrix: 
            return 0

        rows, cols = len(matrix), len(matrix[0])
        heights = [0] * cols
        best = 0
        for i in range(len(matrix)):
            # matrix[i] is the current row
            
            for j in range(len(matrix[i])):
                # matrix[i][j] is "1" or "0"
                if matrix[i][j] == "1":
                    heights[j] += 1
                else:
                    heights[j] = 0
            
            for start_col in range(cols):
                for end_col in range(start_col, cols):
                    # find smallest height between left and right
                    # count columns: right - left + 1
                    # area = min_height * count
                    # best = max(best, area)
                    min_h = float('inf')
                    for k in range(start_col, end_col + 1):
                        if heights[k] < min_h:
                            min_h = heights[k]
                    width = end_col - start_col + 1
                    area = min_h * width
                    if area > best:
                        best = area
                    
        return best

if __name__ == '__main__':
    sol = Solution()
    # Test your cases here
    print(sol.maximalRectangle([["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]))    # Should print: abc
    print(sol.maximalRectangle([["1","0","1","0","0"],["1","1","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]])) # Should print: acdb

6
10


ok so then what if it is
10100
01011
11111
10101
010101
11111

then what would it? Because somehow after row 4 you have 4, 0, 0, 3, 0, meaning whenever there is a 0 we start over, which makes sense now considering we need to be consecutive, but that just shows consecutive for the first col

Yes, and the important correction is this: the “consecutive” part is tracked independently for every column, not just the first one.

So after each row, heights[j] means:

“How many 1s have I seen in a row in column j, ending at this row?”
That is why a 0 resets only that column. It does not mean the whole rectangle is only about the first column.

Using your first four rows, the height arrays would be:

Row 1: 10100 → [1, 0, 1, 0, 0]
Row 2: 01011 → [0, 1, 0, 1, 1]
Row 3: 11111 → [1, 2, 1, 2, 2]
Row 4: 10101 → [2, 0, 2, 0, 3]
That last one is the key example. It does not mean “only the first column matters.” It means:

column 1 has a vertical streak of 2
column 3 has a streak of 2
column 5 has a streak of 3
Then you check horizontal ranges like:

columns 1 to 1 → height 2, width 1, area 2
columns 3 to 3 → height 2, width 1, area 2
columns 5 to 5 → height 3, width 1, area 3
columns 1 to 3 → minimum height is 0 because column 2 is 0, so that range cannot form a rectangle of 1s
So the histogram step is not “find one column and keep going.” It is “build a bar chart for this row, then ask which adjacent columns can form a solid rectangle.”

One more thing: the matrix you wrote is not rectangular as written. 010101 has 6 characters, while the others have 5. For this problem, all rows need the same length. If you want, I can take your exact example and show the largest rectangle row by row once you confirm whether that fifth row should be 01010 or 010101.

In [ ]:
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        # and we got this from counting consecutive 
        # 1s in each col going row by row.

## The Monotonic Stack Approach

**Problem:** Find the largest rectangle in a histogram in O(m) time instead of O(m³).

**Key Idea:** Use a stack to remember which bars are "useful" candidates for rectangles. When a shorter bar arrives, the tall bars before it can no longer extend right, so calculate their areas and remove them.

### Visual Example: Histogram Heights [2, 1, 5, 6, 2, 3]

Let me walk through this step by step using a simple diagram.

```
Histogram bars (indices 0-5):
    6  
    5  
  5 |  
  | | 6
2 | | |
| | 1 | |
+-+-+-+-+-+-+
0 1 2 3 4 5   <- indices

heights = [2, 1, 5, 6, 2, 3]
```

**The Algorithm:**
- Walk left to right through each bar
- Keep a stack of bar **indices** (not heights)
- The stack stays in increasing height order
- When you see a shorter bar:
  - Pop the taller bars (they can't go right anymore)
  - Calculate their max rectangle area
  - Push the current bar

### Step-by-Step Walkthrough

**Start:** stack = [], best_area = 0

**Step 1: height=2 (index 0)**
```
Stack stores indices in increasing height order.
Stack: [0]        heights: [2]
```

**Step 2: height=1 (index 1)**
```
1 < 2, so 2 can't extend right.
Pop index 0. 
  Bar at index 0 has height 2.
  It can extend from index 0 to index 0 (width 1).
  Area = 2 × 1 = 2
  
Stack: [1]        heights: [1]
best_area = 2
```

**Step 3: height=5 (index 2)**
```
5 > 1, so just push.
Stack: [1, 2]        heights: [1, 5]
```

**Step 4: height=6 (index 3)**
```
6 > 5, so just push.
Stack: [1, 2, 3]        heights: [1, 5, 6]
```

**Step 5: height=2 (index 4)**
```
2 < 6, so pop bar at index 3.
  Bar at index 3 has height 6.
  It can extend from index 3 to index 3 (width 1).
  Area = 6 × 1 = 6

Stack: [1, 2]        heights: [1, 5]

2 < 5, so pop bar at index 2.
  Bar at index 2 has height 5.
  **KEY:** Its left boundary is index 1+1 = 2 (next bar in stack is at index 1).
  Its right boundary is index 4-1 = 3 (current bar at index 4).
  Width = 3 - 2 + 1 = 2
  Area = 5 × 2 = 10
  
Stack: [1, 4]        heights: [1, 2]
best_area = 10
```

**Step 6: height=3 (index 5)**
```
3 > 2, so just push.
Stack: [1, 4, 5]        heights: [1, 2, 3]
```

**End of input:** Pop remaining bars.
```
Pop index 5: height 3, width 1, area 3
Pop index 4: height 2, width 3 (from index 2 to 4), area 6
Pop index 1: height 1, width 6 (entire width), area 6
```

**Result:** best_area = **10** (the 5-width-2 rectangle)

---

### Why This Works

The stack ensures:
1. **When you pop a bar,** the bar below it in the stack is the **first shorter bar to the left**
2. **The current bar you're looking at** is the **first shorter bar to the right**
3. So the width is known without rescanning

This eliminates the O(m²) nested loop!


In [1]:
def largestRectangleInHistogram(heights):
    """
    Find the largest rectangle area in a histogram using monotonic stack.
    
    Example:
        heights = [2, 1, 5, 6, 2, 3]
        Output: 10 (the 5-width-2 rectangle from indices 2-3)
    """
    stack = []  # Stack stores INDICES, not heights
    max_area = 0
    
    print(f"\nHistogram heights: {heights}\n")
    
    for i, h in enumerate(heights):
        print(f"--- Process bar {i} (height={h}) ---")
        
        # Pop bars taller than current bar
        # These bars can't extend right anymore
        start = i  # Left boundary for bars we're popping
        
        while stack and heights[stack[-1]] > h:
            popped_index = stack.pop()
            popped_height = heights[popped_index]
            
            # The bar we just popped:
            # - Can't extend right: blocked by current bar at index i
            # - Can't extend left: blocked by bar below it in stack (or start from 0)
            width = i - start  # Distance from left boundary to right boundary (current bar)
            area = popped_height * width
            
            left_bound = stack[-1] + 1 if stack else 0  # Left boundary
            right_bound = i - 1  # Right boundary
            
            print(f"  Pop index {popped_index}: height={popped_height}, width={width}, area={area}")
            print(f"    (extends from col {left_bound} to col {right_bound})")
            
            max_area = max(max_area, area)
            start = popped_index  # Shrink the left boundary
        
        stack.append(i)
        print(f"  Push index {i}. Stack indices: {stack}, Stack heights: {[heights[idx] for idx in stack]}")
    
    # Don't forget bars left in stack at the end
    print(f"\n--- End of input. Pop remaining bars ---")
    while stack:
        popped_index = stack.pop()
        popped_height = heights[popped_index]
        width = len(heights) - start  # Width extends to the end
        area = popped_height * width
        
        left_bound = stack[-1] + 1 if stack else 0
        right_bound = len(heights) - 1
        
        print(f"  Pop index {popped_index}: height={popped_height}, width={width}, area={area}")
        print(f"    (extends from col {left_bound} to col {right_bound})")
        
        max_area = max(max_area, area)
        start = popped_index
    
    print(f"\nMax area: {max_area}\n")
    return max_area

# Test with the example
largestRectangleInHistogram([2, 1, 5, 6, 2, 3])



Histogram heights: [2, 1, 5, 6, 2, 3]

--- Process bar 0 (height=2) ---
  Push index 0. Stack indices: [0], Stack heights: [2]
--- Process bar 1 (height=1) ---
  Pop index 0: height=2, width=0, area=0
    (extends from col 0 to col 0)
  Push index 1. Stack indices: [1], Stack heights: [1]
--- Process bar 2 (height=5) ---
  Push index 2. Stack indices: [1, 2], Stack heights: [1, 5]
--- Process bar 3 (height=6) ---
  Push index 3. Stack indices: [1, 2, 3], Stack heights: [1, 5, 6]
--- Process bar 4 (height=2) ---
  Pop index 3: height=6, width=0, area=0
    (extends from col 3 to col 3)
  Pop index 2: height=5, width=1, area=5
    (extends from col 2 to col 3)
  Push index 4. Stack indices: [1, 4], Stack heights: [1, 2]
--- Process bar 5 (height=3) ---
  Push index 5. Stack indices: [1, 4, 5], Stack heights: [1, 2, 3]

--- End of input. Pop remaining bars ---
  Pop index 5: height=3, width=1, area=3
    (extends from col 5 to col 5)
  Pop index 4: height=2, width=1, area=2
    (extends 

5

## Integrating Stack Back into Matrix Solution

The maximal rectangle algorithm stays the same:
1. Build histogram for each row (update heights array)
2. **Instead of O(m³) nested loops**, call the stack-based histogram function in O(m) time

**Before (slow):**
```
for each row:
    update heights
    for left in range(cols):          # O(m)
        for right in range(left, cols):  # O(m²)
            find min in [left, right]    # O(m³)
            calculate area
```

**After (fast):**
```
for each row:
    update heights
    max_area_this_row = stack_histogram(heights)  # O(m) instead of O(m³)
```

Total: O(n × m) instead of O(n × m³)

### Optimization Benefit

For a 100×100 matrix:
- **Naive:** 100 × 100³ = 100,000,000 operations
- **Stack:** 100 × 100 = 10,000 operations
- **Speed-up: 10,000×**


In [5]:
class OptimizedSolution:
    """
    Maximal Rectangle using Monotonic Stack
    Time: O(n × m) instead of O(n × m³)
    Space: O(m) for the heights array and stack
    """
    
    def largestRectangleInHistogram(self, heights):
        """
        Solve histogram in O(m) using monotonic stack.
        
        Stack stores indices. When we pop, we know:
        - Right boundary: current index i (first bar shorter than popped bar)
        - Left boundary: bar below popped bar in stack (or 0 if empty)
        """
        stack = []
        max_area = 0
        
        for i, h in enumerate(heights):
            # Pop all bars taller than current bar
            while stack and heights[stack[-1]] > h:
                popped_idx = stack.pop()
                popped_height = heights[popped_idx]
                
                # Left boundary: next bar in stack, or 0 if empty
                left_idx = stack[-1] if stack else -1
                # Right boundary: current bar (the first shorter one)
                right_idx = i - 1
                
                width = right_idx - left_idx
                area = popped_height * width
                max_area = max(max_area, area)
            
            # Push current bar
            stack.append(i)
        
        # Pop remaining bars: they extend to the end
        while stack:
            popped_idx = stack.pop()
            popped_height = heights[popped_idx]
            
            left_idx = stack[-1] if stack else -1
            right_idx = len(heights) - 1
            
            width = right_idx - left_idx
            area = popped_height * width
            max_area = max(max_area, area)
        
        return max_area
    
    def maximalRectangle(self, matrix):
        if not matrix:
            return 0
        
        rows, cols = len(matrix), len(matrix[0])
        heights = [0] * cols
        best = 0
        
        for i in range(rows):
            # Update histogram
            for j in range(cols):
                if matrix[i][j] == "1":
                    heights[j] += 1
                else:
                    heights[j] = 0
            
            # Find max rectangle in this histogram using stack
            area = self.largestRectangleInHistogram(heights)
            best = max(best, area)
        
        return best

# Test both solutions
test1 = [
    ["1","0","1","0","0"],
    ["1","0","1","1","1"],
    ["1","1","1","1","1"],
    ["1","0","0","1","0"]
]

test2 = [
    ["1","0","1","0","0"],
    ["1","1","1","1","1"],
    ["1","1","1","1","1"],
    ["1","0","0","1","0"]
]

naive = NaiveSolution()
opt = OptimizedSolution()

print("Test 1:")
print(f"  Naive:     {naive.maximalRectangle(test1)}")
print(f"  Optimized: {opt.maximalRectangle(test1)}")

print("\nTest 2:")
print(f"  Naive:     {naive.maximalRectangle(test2)}")
print(f"  Optimized: {opt.maximalRectangle(test2)}")

Test 1:
  Naive:     6
  Optimized: 6

Test 2:
  Naive:     10
  Optimized: 10


## Summary: How the Stack Optimizes Histogram

| Approach | Inner Logic | Time | Why? |
|----------|------------|------|------|
| **Naive (original)** | For each pair (left, right), scan all bars to find min | O(m³) per row | Triple nested loop: left, right, scan |
| **Stack (optimized)** | Each bar pushed/popped once, width known instantly | O(m) per row | When popping, boundaries are already known from stack |

### Visual Comparison on a 100×100 Matrix

```
Naive:    100 rows × 100³ operations = 100,000,000 ops
Stack:    100 rows × 100 operations = 10,000 ops

Speed-up: 10,000× faster!
```

### Key Stack Insight

When you pop a bar from the stack:
- **Right boundary** = current index (first shorter bar to the right)
- **Left boundary** = bar below it in stack (first shorter bar to the left)
- **Width** = right - left automatically computed with no rescanning!

This is the core optimization: you never rescan—the stack structure gives you the boundaries for free.

---

### What You Now Understand

✓ Histogram representation of the matrix problem  
✓ Why naive O(m³) checking all ranges is slow  
✓ How a monotonic stack tracks useful bars  
✓ Why popping bars calculates their max rectangle instantly  
✓ The O(m) speed-up in action  

The stack transforms a brute-force "check everything" approach into an elegant "remember only what you need" approach. This pattern appears in many hard problems!


In [4]:
# Compare: run your ORIGINAL naive solution
class NaiveSolution:
    def maximalRectangle(self, matrix):
        if not matrix: 
            return 0

        rows, cols = len(matrix), len(matrix[0])
        heights = [0] * cols
        best = 0
        for i in range(len(matrix)):
            for j in range(len(matrix[i])):
                if matrix[i][j] == "1":
                    heights[j] += 1
                else:
                    heights[j] = 0
            
            for start_col in range(cols):
                for end_col in range(start_col, cols):
                    min_h = float('inf')
                    for k in range(start_col, end_col + 1):
                        if heights[k] < min_h:
                            min_h = heights[k]
                    width = end_col - start_col + 1
                    area = min_h * width
                    if area > best:
                        best = area
                    
        return best

# Test both
test1 = [
    ["1","0","1","0","0"],
    ["1","0","1","1","1"],
    ["1","1","1","1","1"],
    ["1","0","0","1","0"]
]

naive = NaiveSolution()
opt = OptimizedSolution()

print(f"Test 1 - Naive: {naive.maximalRectangle(test1)}")
print(f"Test 1 - Optimized: {opt.maximalRectangle(test1)}")


Test 1 - Naive: 6
Test 1 - Optimized: 2


```markdown
Column:
Row 0: 1  → height = 1
Row 1: 1  → height = 2  (yes! last two rows are both 1)
Row 2: 0  → height = 0  (reset)
Row 3: 1  → height = 1
Row 4: 1  → height = 2  (yes! last two rows are both 1)
```

At row 4, height = 2 means:

Row 3 has a 1 in this column
Row 4 has a 1 in this column
Together they form a vertical stack of 2
What This Means for Rectangles
A height of 2 in multiple columns means those columns can form a 2-row-tall rectangle (using the last 2 rows).

For example, if at row 4 you have heights [2, 0, 2, 0, 3]:

Column 0: 2-row stack (rows 3-4)
Column 2: 2-row stack (rows 3-4)
Column 4: 3-row stack (rows 2-3-4)
So you could make a rectangle using columns 0 and 2, height 2, width 2 → area 4.

Or a rectangle using just column 4, height 3, width 1 → area 3.

The heights array at any moment tells you: "How far down can each column reach with consecutive 1s?"

# Summary - In my words
So we need to find the area of a rectangle containing only 1s, so it needs to have an area that would be equal to all 1s for its size.

We need the length and the width of the rectangle, and in order to find that we need to know the left boundary, the right boundary, (beginning/end) and the height.  As we will know if it starts col 2, and end col 5, with a height of 2, the area is then right - left + 1.  
The + 1 is because of the index starting at 0.  
The right - left is to find the width.
Length then is the height

We use **min_height** because the rectangle will be the size or height of the min of the heights.

## Key Insight: The Minimum Height Constraint

With heights = [3, 1, 1, 3]:

```
Height 3:  |#|   |#|
Height 2:  |#|   |#|
Height 1:  |#|#|#|#|
           +-------+
           0 1 2 3
```

**The only solid rectangle you can make is 1×4** (height=1, width=4, area=4).

Why? Because columns 1 and 2 only go up to height 1. If you tried height 2 or 3, you'd be including empty space (0s) in columns 1 and 2—violating the "all 1s" rule.

**The minimum height (1) is the bottleneck.**

This is exactly why we search for `min_h` in our range. The minimum bar in any range determines the maximum height that rectangle can be.

### Different ranges, same principle:
- Columns 0-3: min = 1 → area = 1 × 4 = 4
- Columns 0-2: min = 1 → area = 1 × 3 = 3  
- Columns 0-1: min = 1 → area = 1 × 2 = 2
- Columns 1-2: min = 1 → area = 1 × 2 = 2

The minimum always limits you.